In [4]:
# Chipathon 2026 - NYCMOS
# This notebook computes the sizing for Gilbert cell mixer switching pair & LO quad based on gm/Id method
# Resource: https://github.com/bmurmann/Chipathon2025/tree/main

from pygmid import Lookup as lk
from scipy.io import loadmat
import numpy as np
import pandas as pd
import os

In [5]:
# Change file depending on where you cloned repo
n = lk("../lookup_tables/nfet_03v3.mat") #located in NYCMOS/lookup_tables/nfet_03v3.mat, replaced NYCMOS with .. because idk how to get paths to work in jupyter
p = lk('../lookup_tables/pfet_03v3.mat')
print(n._Lookup__DATA.keys())
print(n._Lookup__DATA.keys())

dict_keys(['INFO', 'CORNER', 'TEMP', 'VGS', 'VDS', 'VSB', 'L', 'W', 'NFING', 'ID', 'VT', 'GM', 'GMB', 'GDS', 'CGG', 'CGB', 'CGD', 'CGS', 'CDD', 'CSS', 'STH', 'SFL'])
dict_keys(['INFO', 'CORNER', 'TEMP', 'VGS', 'VDS', 'VSB', 'L', 'W', 'NFING', 'ID', 'VT', 'GM', 'GMB', 'GDS', 'CGG', 'CGB', 'CGD', 'CGS', 'CDD', 'CSS', 'STH', 'SFL'])


In [6]:
# Target for output amplifier

In [11]:
# Target
Rout = 50 # target output resistance
Rs_est = 100 # RL is parallel with RS and so RS is 50 or smaller
gm_required = (Rs_est/Rout-1)/Rs_est * np.array([1, 1, 1]) # rout = (1/gm) || Rs
# gm_required = np.array([10e-3, 10e-3, 10e-3])
gm_id = np.array([5, 10, 15]) #weak to moderate inversion
l = 0.28
print(gm_required)
print(10e-3)

[0.01 0.01 0.01]
0.01


In [12]:
# Sizing for output transistor

id_required = gm_required / gm_id #required Id for the gm target and gm_id considered
Jd = n.lookup('ID_W', GM_ID=gm_id, L = l) # A/um current density for gm_id considered

print(Jd)

W = id_required / Jd # width required to match id_required
cgg_w = n.lookup('CGG_W', GM_ID=gm_id, L = l) # cgg/w for the given width

cgg = cgg_w * W # convert to absolute capacitance
ft = gm_required / (2 * np.pi * cgg) # Estimate transit freq

vdsat_est = 2/gm_id 

print(gm_required.shape)

df = pd.DataFrame(
    [gm_required, gm_id, id_required*1e3, Jd, W, cgg, ft, vdsat_est],
    index=['gm_required', 'gm/id', 'id_required (mA)', 'Jd', 'W (um)', 'Cgg', 'ft', 'Vdsat'],
    columns = ['1','2','3']
)
df

[2.53926964e-05 6.84724166e-06 2.09748172e-06]
(3,)


,1,2,3
gm_required,1.000000e-02,1.000000e-02,1.000000e-02
gm/id,5.000000e+00,1.000000e+01,1.500000e+01
id_required (mA),2.000000e+00,1.000000e+00,6.666667e-01
Jd,2.539270e-05,6.847242e-06,2.097482e-06
W (um),7.876281e+01,1.460442e+02,3.178415e+02
Cgg,9.257212e-14,1.603948e-13,3.253565e-13
ft,1.719253e+10,9.922699e+09,4.891709e+09
Vdsat,4.000000e-01,2.000000e-01,1.333333e-01
